# Comandos Básicos de Pub Sub con Python

La biblioteca ***google-cloud-pubsub*** de Python es una interfaz para interactuar con Google Cloud Pub/Sub, un servicio de mensajería asíncrona que permite a los desarrolladores enviar y recibir mensajes entre aplicaciones independientes. Aquí te explico algunas de las funciones más utilizadas junto con ejemplos de cómo implementarlas:

In [6]:
!pip install google-cloud-pubsub

You should consider upgrading via the 'C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip' command.


In [ ]:
# from google.colab import auth
# auth.authenticate_user()

**1. Inicialización del Cliente**

Para interactuar con Google Cloud Pub/Sub, primero necesitas crear un cliente.

In [7]:
from google.cloud import pubsub_v1

publisher_client = pubsub_v1.PublisherClient()
subscriber_client = pubsub_v1.SubscriberClient()
project_id = 'project-d0c76d9f-d27a-4298-8e3'
my_topic = 'my-topic'
my_subs = 'my-subscription'

**2. Crear un Tópico**

Un tópico es un canal a través del cual se envían los mensajes.

In [ ]:
topic_path = publisher_client.topic_path(project_id, my_topic)
topic = publisher_client.create_topic(request={"name": topic_path})

**3. Publicar Mensajes**

Puedes publicar mensajes en un tópico. Los mensajes deben estar codificados en bytes.

In [7]:
data = 'Hello, World 19!'
data = data.encode('utf-8')
future = publisher_client.publish(topic_path, data)
print(future.result())

19252649473727194


**4. Suscribirse a un Tópico**

Para recibir mensajes, necesitas suscribirte a un tópico creando una suscripción.

In [8]:
subscription_path = f'projects/{project_id}/subscriptions/{my_subs}'


In [ ]:
subscription = subscriber_client.create_subscription(request={"name": subscription_path, "topic": topic_path})

**5. Recibir Mensajes**

Una vez suscrito, puedes comenzar a recibir mensajes. Los mensajes son recibidos en un callback.

In [ ]:
import time

def callback(message):
    # Decode the message data to string
    message_data = message.data.decode('utf-8')
    print(f'Received message: {message_data}')
    message.ack()

# Subscribe to the topic
streaming_pull_future = subscriber_client.subscribe(subscription_path, callback=callback)
print(f'Listening for messages on {subscription_path}...')

try:
    # Keep the main thread alive to listen for messages
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    # Stop the subscriber when a KeyboardInterrupt is received
    streaming_pull_future.cancel()
    streaming_pull_future.result()  # Block until the shutdown is complete

Listening for messages on projects/project-d0c76d9f-d27a-4298-8e3/subscriptions/my-subscription...


**6. Eliminar un Tópico o Suscripción**

Cuando ya no los necesites, puedes eliminar tanto tópicos como suscripciones.

Eliminar una suscripción:

In [9]:
request = pubsub_v1.types.DeleteSubscriptionRequest(subscription=subscription_path)
subscriber_client.delete_subscription(request=request)

Eliminar un tópico:

In [10]:
publisher_client.delete_topic(request={"topic": topic_path})